## 1 · Imports

In [ ]:
# import scvelo as scv
from multiprocessing import Pool

import anndata as ad
import matplotlib as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

In [ ]:
import mudata as mu

In [ ]:
import os
from functools import partial
from pathlib import Path
from typing import Optional

import altair as alt

# Libraries
import anndata as ad
import decoupler as dc
import matplotlib as plt
import mudata as mu
import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import seaborn as sns
from anndata import AnnData
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation

alt.data_transformers.enable("vegafusion")

import anndata as ad
import numpy as np
import palantir
import scipy.sparse as sp

## 2 · Data Loading

Load mudata

In [ ]:
mdata = mu.read_h5mu("001_create_mudata_normal.h5mu")

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["GF"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["ctrl"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["effector"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["GF1"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["GF2"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["ctrl1"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["ctrl2"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["effector1"])

In [ ]:
sc.pl.umap(mdata["gex"], color=["sample_id", "condition"], groups=["effector2"])

In [ ]:
sc.pl.umap(
    mdata["gex"], color=["rna_leiden"], legend_loc="on data"
)  # , groups = ["7"])

In [ ]:
sc.tl.leiden(mdata["gex"], resolution=1.2, key_added="rna_leiden_12")

In [ ]:
sc.pl.umap(
    mdata["gex"], color=["rna_leiden_12"], legend_loc="on data"
)  # , groups = ["7"])

## 4 · Gene marker ploting


In [ ]:
import scanpy as sc


def plot_marker_umaps(adata, marker_dict, title_prefix):
    for category, genes in marker_dict.items():
        genes = sorted(list(genes))
        genes_present = [g for g in genes if g in adata.var_names]

        if len(genes_present) == 0:
            print(f"No genes found for {category}")
            continue

        print(f"Plotting {category}: {genes_present}")

        sc.pl.umap(
            adata,
            color=genes_present,
            ncols=4,
            cmap="Reds",
            vmax="p99",
            title=[f" {category} | {g}" for g in genes_present],
            frameon=False,
        )

In [ ]:
gene_markers = {
    "Naive": {
        "Sell",
        "Ccr7",  #  \(CD44^{lo}\), \(CD62L^{hi}\), \(CCR7^{hi}\)
        "Cd44",
    },    "Activated": {
        "Cd69",
        "Nr4a1",
        "Nr4a2",
        "Nr4a3",
        "Fos",
        "Jun",
        "Egr1",
        "Egr2",

        "Irf4",
        "Batf",
       
    },
    "Tissue resident memory": {
        "Itgae",
        "Itga1",  # CD69, CD103, and CD49a.
        "Cxcr6",
        "Zfp683",  # Hobit

        "Rgs1",
        "Ahr",
        "Fabp5","Gzmk",
        "Gzmb",
    },


    "Cytotoxic/Effector memory": {"Gzmk",        "Runx3",
        "Gzmb",
        "Prf1",
        "Nkg7",
        "Ifng",
        "Fasl",
        "Ifng",
        "Ccl5",

                                
    },
}

In [ ]:
plot_marker_umaps(
    adata=mdata["gex"], marker_dict=gene_markers, title_prefix="Gene markers"
)

In [ ]:
from matplotlib import cm

sc.pl.dotplot(
    mdata["gex"],
    gene_markers,
    groupby="rna_leiden_12",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,  # ,save="annotation.svg"
)


## naive #  \(CD44^{lo}\), \(CD62L^{hi}\), \(CCR7^{hi}\)

In [ ]:
from matplotlib import cm

sc.pl.matrixplot(
    mdata["gex"],
    gene_markers,
    groupby="rna_leiden_12",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,  # ,save="annotation.svg"
)

In [ ]:
sc.pl.umap(
    mdata["gex"], color=["rna_leiden_12"], legend_loc="on data"
)  # , groups = ["7"])

## 5 · Annotation


In [ ]:
# Check unique clusters first (optional, but recommended)
print(mdata["gex"].obs["rna_leiden_12"].unique())

# Create mapping dictionary (adjust numbers if needed)
cluster_annotation = {
    "0": "Naive",
    "1": "Naive",
    "2": "Activated",
    "3": "Cytotoxic/Effector memory",
    "4": "Naive",
    "5": "Cytotoxic/Effector memory",
    "6": "Tissue resident memory",
    "7": "Tissue resident memory",
    "8":"Tissue resident memory",
    "9":"Cytotoxic/Effector memory",
    "10":"Activated"
}

# If gex03 is integer instead of string, remove the quotes in dictionary keys
# e.g., 0: "Activated"

# Create new annotation column
mdata["gex"].obs["cell_annotation_05"] = (
    mdata["gex"].obs["rna_leiden_12"].map(cluster_annotation)
)

# Make it categorical (recommended for plotting)
mdata["gex"].obs["cell_annotation_05"] = (
    mdata["gex"].obs["cell_annotation_05"].astype("category")
)

# Verify
print(mdata["gex"].obs[["rna_leiden_12", "cell_annotation_05"]].head())

In [ ]:
sc.pl.umap(
    mdata["gex"], color="cell_annotation_05", frameon=False  # , legend_loc="on data"
)

In [ ]:
mdata["gex"].obs.cell_annotation_05.unique()

## 6 · Dotplot


In [ ]:
desired_order = ["Naive","Activated",
    "Tissue resident memory", "Cytotoxic/Effector memory",

]

mdata["gex"].obs["cell_annotation_05"] = (
    mdata["gex"]
    .obs["cell_annotation_05"]
    .astype("category")
    .cat.reorder_categories(desired_order, ordered=True)
)

sc.pl.dotplot(
    mdata["gex"],
    gene_markers,
    groupby="cell_annotation_05",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,
)

In [ ]:
sc.pl.matrixplot(
    mdata["gex"],
    gene_markers,
    groupby="cell_annotation_05",
    standard_scale="var",
    dendrogram=False,
    swap_axes=False,
)

In [ ]:
sc.pl.umap(
    mdata["gex"], color="cell_annotation_05", frameon=False  # , legend_loc="on data"
)

In [ ]:
mdata.write_h5mu("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/002_annotate_mudata_normal.h5mu")